In [2]:
# read the input datset
with open('shakesphere_input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("length if dataset: ", len(text))

length if dataset:  1115394


In [6]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [1]:
uniqu_chars = sorted(set(text))
vocab_size = len(uniqu_chars)
print(len(uniqu_chars))
print(''.join(uniqu_chars))

NameError: name 'text' is not defined

In [13]:
stoi = { ch:i for i,ch in enumerate(uniqu_chars)}
itos = { i:ch for i,ch in enumerate(uniqu_chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello there!"))
print(decode(encode("hello there!")))


[46, 43, 50, 50, 53, 1, 58, 46, 43, 56, 43, 2]
hello there!


In [ ]:
#encode entire data set and store it into a toch.tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

In [22]:
# seprate text into train and test split
n = int(0.9*len(text))
train_data= data[:n]
val_data = data[n:]
if len(train_data) + len(val_data) == len(text):
    print("train and test split looks good!") 

train and test split looks good!


In [24]:
torch.manual_seed(1657)
batch_size = 4 #how many individual sequences we shall process in parallel
block_size = 8 # what is the max context len for predictions


def get_batch(split):
    # genrate a small batch of data of i/ps x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x,y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets')
print(yb.shape)
print(yb)

print('------')

for b in range(batch_size) :
    for t in range(block_size):
        context  = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 5,  1, 47, 44,  1, 42, 43, 39],
        [ 0, 35, 46, 63,  1, 50, 43, 58],
        [42,  1, 45, 47, 60, 43,  1, 59],
        [49,  1, 46, 47, 51, 11,  0, 25]])
targets
torch.Size([4, 8])
tensor([[ 1, 47, 44,  1, 42, 43, 39, 58],
        [35, 46, 63,  1, 50, 43, 58,  1],
        [ 1, 45, 47, 60, 43,  1, 59, 57],
        [ 1, 46, 47, 51, 11,  0, 25, 39]])
------
when input is [5] target: 1
when input is [5, 1] target: 47
when input is [5, 1, 47] target: 44
when input is [5, 1, 47, 44] target: 1
when input is [5, 1, 47, 44, 1] target: 42
when input is [5, 1, 47, 44, 1, 42] target: 43
when input is [5, 1, 47, 44, 1, 42, 43] target: 39
when input is [5, 1, 47, 44, 1, 42, 43, 39] target: 58
when input is [0] target: 35
when input is [0, 35] target: 46
when input is [0, 35, 46] target: 63
when input is [0, 35, 46, 63] target: 1
when input is [0, 35, 46, 63, 1] target: 50
when input is [0, 35, 46, 63, 1, 50] target: 43
when input is [0, 35, 46, 63, 1, 50, 43

In [5]:
%pip install torch
import torch
import torch.nn as nn
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        return logits
    
m = BigramLanguageModel(vocab_size)
out = m(xb,yb)
print(out.shape) # (batch_size, block_size, vocab_size)


^C
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'torch'

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/113.8 MB ? eta -:--:--
   ---------------------------------------- 1.0/113.8 MB 3.6 MB/s eta 0:00:32
    --------------------------------------- 2.4/113.8 MB 4.8 MB/s eta 0:00:24
   - -------------------------------------- 3.7/113.8 MB 5.4 MB/s eta 0:00:21
   - -------------------------------------- 5.2/113.8 MB 5.8 MB/s eta 0:00:19
   -- ------------------------------------- 7.3/113.8 MB 6.6 MB/s eta 0:00:17
   --- ------------------------------------ 9.7/113.8 MB 7.4 MB/s eta 0:00:15
   ---- ----------------------------------- 11.8/113.8 MB 7.8 MB/s eta 0:00:14
   ---- ----------------------------------- 14.2


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
